# RelBench rel-hm (H&M Fashion) Dataset Demo

**FK Links, Cardinality Stats & Parent-Child Feature Pairs**

This notebook demonstrates the extraction and analysis of foreign-key (FK) relationships from the H&M Fashion relational database (RelBench). It loads pre-computed data containing:

- **2 FK link examples**: `transaction→customer` and `transaction→article`
- **Cardinality statistics**: mean/median/max/p95/std and histograms per FK link
- **Aligned parent-child feature pairs**: sampled rows showing how parent and child table features relate across FK joins

All categorical columns are label-encoded, dates converted to epoch seconds, and high-cardinality text columns dropped.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/dataset_iter1_relbench_rel_hm/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub (Colab) or local file (development)."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'])} dataset(s)")
for ds in data['datasets']:
    print(f"  Dataset: {ds['dataset']} — {len(ds['examples'])} FK link(s)")

## Configuration

Tunable parameters for the demo analysis. These are set to minimal values for quick execution.

In [ ]:
# --- Config ---
MAX_SAMPLE_PAIRS = 50       # Max parent-child pairs to analyze per FK link (original: 5000)
N_HISTOGRAM_BINS = 20       # Number of bins for cardinality histograms
N_SCATTER_POINTS = 50       # Max points in scatter plots

## Parse FK Link Examples

Each example contains an `input` (FK link metadata: table names, feature columns, dimensions, row counts) and an `output` (cardinality statistics and sampled aligned parent-child feature pairs).

In [ ]:
# Parse all FK link examples
examples = []
for ds in data['datasets']:
    for ex in ds['examples']:
        inp = json.loads(ex['input'])
        out = json.loads(ex['output'])
        examples.append({
            'metadata': {k: v for k, v in ex.items() if k.startswith('metadata_')},
            'input': inp,
            'output': out,
        })

print(f"Parsed {len(examples)} FK link example(s)\n")
for i, ex in enumerate(examples):
    inp = ex['input']
    out = ex['output']
    print(f"--- FK Link {i+1}: {inp['child_table']}.{inp['fk_column']} → {inp['parent_table']} ---")
    print(f"  Parent: {inp['parent_table']} ({inp['parent_row_count']} rows, {inp['parent_feature_dim']} features)")
    print(f"  Child:  {inp['child_table']} ({inp['child_row_count']} rows, {inp['child_feature_dim']} features)")
    print(f"  Parent features: {inp['parent_feature_columns']}")
    print(f"  Child features:  {inp['child_feature_columns']}")
    print(f"  Cardinality: mean={out['mean_cardinality']:.2f}, median={out['median_cardinality']:.0f}, "
          f"max={out['max_cardinality']}, p95={out['p95_cardinality']:.0f}, std={out['std_cardinality']:.2f}")
    print(f"  Sampled pairs: {len(out['sample_parent_child_pairs'])}")
    print()

## Build Feature DataFrames from Sampled Pairs

Convert the sampled parent-child feature pairs into pandas DataFrames for analysis. Each pair contains aligned parent and child feature vectors from FK-joined rows.

In [ ]:
# Build DataFrames for each FK link's sampled pairs
fk_dataframes = {}
for i, ex in enumerate(examples):
    inp = ex['input']
    out = ex['output']
    pairs = out['sample_parent_child_pairs'][:MAX_SAMPLE_PAIRS]

    parent_cols = [f"parent__{c}" for c in inp['parent_feature_columns']]
    child_cols = [f"child__{c}" for c in inp['child_feature_columns']]

    parent_data = [p['parent_features'] for p in pairs]
    child_data = [p['child_features'] for p in pairs]

    parent_df = pd.DataFrame(parent_data, columns=parent_cols)
    child_df = pd.DataFrame(child_data, columns=child_cols)
    aligned_df = pd.concat([parent_df, child_df], axis=1)

    link_name = f"{inp['child_table']}.{inp['fk_column']}→{inp['parent_table']}"
    fk_dataframes[link_name] = {
        'aligned': aligned_df,
        'parent': parent_df,
        'child': child_df,
        'input': inp,
        'output': out,
    }
    print(f"FK Link: {link_name}")
    print(f"  Aligned DataFrame: {aligned_df.shape}")
    print(aligned_df.describe().round(3).to_string())
    print()

## Cardinality Statistics Summary

Display the cardinality distribution for each FK link — how many child rows map to each parent.

In [ ]:
# Cardinality statistics table
card_rows = []
for link_name, fk in fk_dataframes.items():
    out = fk['output']
    card_rows.append({
        'FK Link': link_name,
        'Unique Parents': out['num_unique_parents'],
        'Total Children': out['num_children_total'],
        'Mean Card.': out['mean_cardinality'],
        'Median Card.': out['median_cardinality'],
        'Max Card.': out['max_cardinality'],
        'P95 Card.': out['p95_cardinality'],
        'Std Card.': out['std_cardinality'],
    })

card_df = pd.DataFrame(card_rows)
print("=== Cardinality Statistics ===")
print(card_df.to_string(index=False))

## Cross-Table Feature Correlations

Compute correlations between parent and child features across the FK join. High cross-table correlations indicate features that are predictable across the relational link — key for message-passing in relational learning.

In [ ]:
# Compute cross-table correlations for each FK link
for link_name, fk in fk_dataframes.items():
    aligned = fk['aligned']
    parent_cols = [c for c in aligned.columns if c.startswith('parent__')]
    child_cols = [c for c in aligned.columns if c.startswith('child__')]

    # Full correlation matrix, then extract cross-table block
    full_corr = aligned.corr()
    cross_corr = full_corr.loc[parent_cols, child_cols]

    print(f"=== Cross-Table Correlations: {link_name} ===")
    print(cross_corr.round(3).to_string())
    print(f"\nMax absolute cross-correlation: {cross_corr.abs().max().max():.4f}")
    print()

## Visualization

**Top row**: Cardinality histograms for each FK link (log-scale counts).
**Bottom row**: Cross-table correlation heatmaps showing parent vs. child feature predictability.

In [ ]:
n_links = len(fk_dataframes)
fig, axes = plt.subplots(2, n_links, figsize=(6 * n_links, 10))
if n_links == 1:
    axes = axes.reshape(2, 1)

for col_idx, (link_name, fk) in enumerate(fk_dataframes.items()):
    out = fk['output']
    inp = fk['input']
    aligned = fk['aligned']

    # --- Top row: Cardinality histogram ---
    ax = axes[0, col_idx]
    hist = out['cardinality_histogram']
    bins = hist['bins']
    counts = hist['counts']
    bin_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(counts))]
    bin_widths = [bins[i+1] - bins[i] for i in range(len(counts))]
    ax.bar(bin_centers, counts, width=bin_widths, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_yscale('log')
    ax.set_xlabel('Cardinality (children per parent)')
    ax.set_ylabel('Count (log scale)')
    ax.set_title(f'Cardinality: {link_name}')
    ax.text(0.95, 0.95, f"mean={out['mean_cardinality']:.1f}\nmedian={out['median_cardinality']:.0f}\nmax={out['max_cardinality']}",
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # --- Bottom row: Cross-table correlation heatmap ---
    ax = axes[1, col_idx]
    parent_cols = [c for c in aligned.columns if c.startswith('parent__')]
    child_cols = [c for c in aligned.columns if c.startswith('child__')]
    cross_corr = aligned.corr().loc[parent_cols, child_cols]

    # Clean up labels for display
    row_labels = [c.replace('parent__', '') for c in parent_cols]
    col_labels = [c.replace('child__', '') for c in child_cols]

    im = ax.imshow(cross_corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_title(f'Cross-Table Corr: {link_name}')
    ax.set_xlabel('Child features')
    ax.set_ylabel('Parent features')

    # Annotate cells
    for r in range(len(row_labels)):
        for c in range(len(col_labels)):
            val = cross_corr.values[r, c]
            ax.text(c, r, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(val) > 0.5 else 'black')

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('fk_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved visualization to fk_analysis.png")

# RelBench rel-hm (H&M Fashion) Dataset: FK Links, Cardinality Stats & Feature Pairs

This notebook demonstrates how to extract **foreign-key (FK) link metadata**, **cardinality statistics**, and **aligned parent-child feature pairs** from the H&M Fashion relational database (RelBench `rel-hm`).

**What this artifact does:**
1. Loads pre-extracted FK link examples (transaction→customer, transaction→article)
2. Parses per-link metadata: table names, feature columns, dimensions, row counts
3. Computes and visualizes cardinality statistics (mean/median/max/p95/std/histogram)
4. Examines sampled aligned parent-child feature pairs for downstream R² cross-table predictability